# MVTec AD — Experiment Results
**BAP: XAI-Supported Evaluation of Model Degradation in Visual Inspection**  
Marwan Elkhallouki · HOGENT AI & Data Engineering

Loads every `output/<category>_<corruption>_results.npz` file and visualises:
1. AUROC degradation curves per corruption type
2. AUROC heat-map (category × severity)
3. Summary statistics table

---
## 0 · Setup

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import warnings
warnings.filterwarnings('ignore')

OUTPUT_DIR = Path('output')

# ── Load all result files ──────────────────────────────────────────────────
pattern = re.compile(r'^(.+?)_(.+?)_results\.npz$')
records = []

for f in sorted(OUTPUT_DIR.glob('*_results.npz')):
    m = pattern.match(f.name)
    if not m:
        continue
    category, corruption = m.group(1), m.group(2)
    d = np.load(f)
    for sev, auc in zip(d['auroc_severities'], d['auroc_values']):
        records.append(dict(category=category, corruption=corruption,
                            severity=int(sev), auroc=float(auc)))

df = pd.DataFrame(records)

CATEGORIES  = sorted(df['category'].unique())
CORRUPTIONS = sorted(df['corruption'].unique())
SEVERITIES  = sorted(df['severity'].unique())

print(f'Loaded {len(df)} data points')
print(f'  Categories  : {CATEGORIES}')
print(f'  Corruptions : {CORRUPTIONS}')
print(f'  Severities  : {SEVERITIES}')
print()
print(df.to_string(index=False))

---
## 1 · AUROC Degradation Curves

In [ ]:
# One subplot per corruption type; one line per category
CAT_COLORS = plt.cm.tab10.colors

n_corr = len(CORRUPTIONS)
fig, axes = plt.subplots(1, n_corr, figsize=(6 * n_corr, 4.5), sharey=True)
if n_corr == 1:
    axes = [axes]

fig.suptitle('AUROC Degradation vs. Corruption Severity', fontsize=14, fontweight='bold')

for ax, corr in zip(axes, CORRUPTIONS):
    sub = df[df['corruption'] == corr]
    for idx, cat in enumerate(CATEGORIES):
        row = sub[sub['category'] == cat].sort_values('severity')
        if row.empty:
            continue
        ax.plot(row['severity'], row['auroc'],
                marker='o', linewidth=2,
                color=CAT_COLORS[idx % len(CAT_COLORS)],
                label=cat.replace('_', ' ').title())
        ax.fill_between(row['severity'], row['auroc'], alpha=0.06,
                        color=CAT_COLORS[idx % len(CAT_COLORS)])

    ax.axhline(0.5, color='grey', linestyle='--', linewidth=1, label='Random (0.50)')
    ax.set_title(corr.replace('_', ' ').title(), fontweight='bold')
    ax.set_xlabel('Corruption severity')
    ax.set_ylabel('AUROC')
    ax.set_ylim(0, 1.05)
    ax.set_xticks(SEVERITIES)
    ax.legend(fontsize=8, frameon=False)

plt.tight_layout()
plt.show()

---
## 2 · AUROC Heat-map (category × severity)

In [ ]:
n_corr = len(CORRUPTIONS)
fig, axes = plt.subplots(1, n_corr, figsize=(5 * n_corr, max(2.5, 0.6 * len(CATEGORIES) + 1.5)))
if n_corr == 1:
    axes = [axes]

fig.suptitle('AUROC Heat-map per Corruption', fontsize=13, fontweight='bold')

cmap = 'RdYlGn'

for ax, corr in zip(axes, CORRUPTIONS):
    sub = df[df['corruption'] == corr]
    matrix = pd.DataFrame(index=CATEGORIES, columns=SEVERITIES, dtype=float)
    for _, row in sub.iterrows():
        matrix.loc[row['category'], row['severity']] = row['auroc']

    im = ax.imshow(matrix.values.astype(float), aspect='auto',
                   cmap=cmap, vmin=0, vmax=1)
    ax.set_xticks(range(len(SEVERITIES)))
    ax.set_xticklabels([f'sev {s}' for s in SEVERITIES])
    ax.set_yticks(range(len(CATEGORIES)))
    ax.set_yticklabels([c.replace('_', ' ').title() for c in CATEGORIES])
    ax.set_title(corr.replace('_', ' ').title(), fontweight='bold')

    for i, cat in enumerate(CATEGORIES):
        for j, sev in enumerate(SEVERITIES):
            val = matrix.loc[cat, sev]
            if pd.notna(val):
                ax.text(j, i, f'{val:.3f}', ha='center', va='center',
                        fontsize=9, fontweight='bold',
                        color='black' if 0.3 < val < 0.75 else 'white')
            else:
                ax.text(j, i, '—', ha='center', va='center', fontsize=9, color='grey')

    plt.colorbar(im, ax=ax, shrink=0.7, label='AUROC')

plt.tight_layout()
plt.show()

---
## 3 · Summary Statistics Table

In [ ]:
# Pivot: rows = category, cols = (corruption, severity)
pivot = df.pivot_table(index='category', columns=['corruption', 'severity'],
                       values='auroc', aggfunc='first')
pivot.columns = [f'{c} / sev{s}' for c, s in pivot.columns]
pivot.index = [c.replace('_', ' ').title() for c in pivot.index]

print('AUROC per category, corruption and severity:')
print(pivot.round(4).to_string())

# Per-category stats
print('\nPer-category summary (across all available results):')
stats = df.groupby('category')['auroc'].agg(['mean', 'min', 'max', 'std']).round(4)
stats.index = [c.replace('_', ' ').title() for c in stats.index]
print(stats.to_string())

---
## 4 · AUROC Drop per Category (severity min → max)

In [ ]:
# For each (category, corruption) pair compute drop from lowest to highest severity
drop_records = []
for (cat, corr), grp in df.groupby(['category', 'corruption']):
    grp = grp.sort_values('severity')
    drop = grp.iloc[0]['auroc'] - grp.iloc[-1]['auroc']
    drop_records.append(dict(category=cat, corruption=corr,
                             auroc_start=grp.iloc[0]['auroc'],
                             auroc_end=grp.iloc[-1]['auroc'],
                             drop=drop))
drop_df = pd.DataFrame(drop_records)

n_corr = len(CORRUPTIONS)
fig, axes = plt.subplots(1, n_corr, figsize=(5 * n_corr, 4), sharey=False)
if n_corr == 1:
    axes = [axes]

fig.suptitle('AUROC Drop (first → last severity)', fontsize=13, fontweight='bold')

for ax, corr in zip(axes, CORRUPTIONS):
    sub = drop_df[drop_df['corruption'] == corr].sort_values('drop', ascending=False)
    colors = ['#EF5350' if d > 0 else '#4CAF50' for d in sub['drop']]
    bars = ax.barh([c.replace('_', ' ').title() for c in sub['category']],
                   sub['drop'], color=colors, edgecolor='white')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(corr.replace('_', ' ').title(), fontweight='bold')
    ax.set_xlabel('AUROC drop (positive = degradation)')
    for bar, (_, row) in zip(bars, sub.iterrows()):
        x = bar.get_width()
        ax.text(x + 0.005 if x >= 0 else x - 0.005,
                bar.get_y() + bar.get_height() / 2,
                f'{x:+.3f}', va='center', fontsize=8,
                ha='left' if x >= 0 else 'right')

plt.tight_layout()
plt.show()